# Design and implementation decisions

## Source and target

The original paper PDF controls over inherited extraction text. Equation (5) is on physical PDF p. 6 (printed p. 40): it regresses working-capital accrual changes in year *t* on past, current, and future operating cash flow, with an intercept. The accompanying text says the residual standard deviation is a firm-level accrual-quality measure and that higher values mean lower quality. The paper does not name the estimator; ordinary least squares (OLS) is used as a best-effort interpretation of “firm-level time-series regression.”

Table 1 and the sample discussion are on physical PDF pp. 7–8 (printed pp. 41–42). The paper names Compustat annual industrial and research files for 1987–1999, requires complete assets, earnings, operating cash flow, and receivables/inventory changes, trims the most extreme 1% of operating cash flow, earnings, and working-capital changes, then requires both leading and lagged cash flows and earnings. The prose on physical p. 8 is broader than Table 1's row label, which names only lead/lag cash from operations. Page 8 defines both constructed Earn (CFO + ΔWC) and Prof (Compustat item 123), and Table 2 includes Earn(t+1). For the firm-specific baseline, I follow the broader sample prose and require CFO and constructed Earn at t−1 and t+1; this source-based interpretation remains uncertain because Table 1 names CFO alone. The paper then requires at least eight usable annual observations per firm. Table 1 reports 59,360 initially available firm-years, 55,850 after trimming, 30,317 after the lead/lag screen, and 15,234 firm-years for 1,725 firms after the eight-observation screen. Those counts are comparison outcomes, not construction targets.

Table 3 Panel A is on physical PDF p. 10 (printed p. 44). Its target is the distribution across firm-specific estimates: mean, t-statistic, and lower quartile/median/upper quartile for each coefficient, plus the corresponding adjusted-R-squared distribution. The table note explicitly says the Panel A t-statistics are based on the distribution of the 1,725 firm-level coefficients. They are not pooled-regression statistics or averages of firm-specific t-statistics. The paper does not specify the individual-fit standard-error correction; none is needed for these cross-firm t-statistics.

## Local data and item mappings

The read-only source is `COMP.FUNDA` (`/storage/sas/base/comp/d_na/funda.sas7bdat`), the catalog-listed merged annual fundamentals file. Local candidate fields and labels were checked against the catalog. The current vintage is not confirmed to reproduce the paper’s historical Compustat population.

The 2004 S&P Research Insight North America Data Guide is the provider documentation in `git/output/reproduction_runs/reference_documents/compustat_research_insight_2004.pdf`; the relevant CURRTR entry extract is retained in `documentation/compustat_2004_p71.txt`. The guide identifies the paper’s legacy annual items as follows:

| Paper item | Provider mnemonic | Local `COMP.FUNDA` field | Provider evidence |
| --- | --- | --- | --- |
| 302, receivables decrease/(increase) | RECCH | `recch` | physical guide p. 875 |
| 303, inventory decrease/(increase) | INVCH | `invch` | physical guide p. 858 |
| 304, accounts payable and accrued liabilities increase/(decrease) | APALCH | `apalch` | physical guide p. 813 |
| 305, income taxes accrued increase/(decrease) | TXACH | `txach` | physical guide p. 851 |
| 307, assets and liabilities—other, net change | AOLOCH | `aoloch` | physical guide p. 818 |
| 308, operating activities—net cash flow | OANCF | `oancf` | physical guide p. 871 |
| 123, income before extraordinary items (cash flow/statement of changes) | IBC | `ibc` | physical guide p. 850 |

The signed working-capital measure follows the paper literally: `-(recch + invch + apalch + txach + aoloch)`. The provider definitions describe these as signed cash-flow statement changes. No sign is reversed a second time. Item 304 is AP and accrued liabilities, not AP alone. Item 307 maps to `aoloch`; the similarly named `uaoloch` is not substituted. Item 123 maps to `ibc` (income before extraordinary items, cash flow/statement of changes), not the local `ib` field. These mappings correct the inherited feasibility note’s uncertainty about the item-to-field crosswalk; they do not prove current-vintage equivalence.

The local merged annual file is restricted to `indfmt='INDL'`, `datafmt='STD'`, `consol='C'`, and `popsrc='D'` as a defensible local definition of the paper’s annual industrial/research population. The source does not report these modern merged-file codes; this population translation is best effort. Raw years 1985–2000 are retained to construct and scale the 1987–1999 outcome window. The analysis key is `gvkey` by `fyear`; duplicate firm-year records and field coverage are checked before any estimation, with no silent `NODUPKEY` resolution.

For currency, the local catalog labels `COMP.FUNDA.curcd` “ISO Currency Code.” The S&P Research Insight guide’s CURRTR entry (physical PDF p. 71; printed p. 21) says the rate is used to translate foreign-currency amounts to U.S. dollars and is the rate in effect at the balance-sheet date. It does not state whether to multiply or divide, nor how that rate should be applied to annual cash-flow items and a two-year average-assets denominator. I therefore do not infer CURRTR arithmetic. The build requires the two source `CURCD` values to be known and identical before it forms any average-assets denominator. Same-currency USD and non-USD observations remain eligible; a denominator spanning unlike or unknown currencies is unavailable. The raw source audit records CURRTR summaries but does not use CURRTR to alter amounts.

## Scaling, timing, and sample screens

The paper states on physical PDF p. 8 that all variables are scaled by average total assets, but it does not explicitly state the denominator date for the lagged and leading CFO terms. This implementation uses each cash-flow observation’s own fiscal-year average assets: `(at_s + at_(s−1))/2` for CFO in fiscal year *s*. Therefore the dependent change in working capital for year *t* and current CFO use the *t* denominator; CFO at *t−1* uses the *t−1* denominator; and CFO at *t+1* uses the *t+1* denominator. This is the natural variable-by-fiscal-year interpretation, but remains an assumption rather than a source-explicit denominator-date rule. Adjacent years are joined by exact `gvkey` and `fyear`; a gap does not count as a lag or lead. A nonpositive or missing average-assets denominator makes the scaled value unavailable. Currency consistency is checked within each denominator: `CURCD_t = CURCD_(t−1)` for the current outcome and CFO(t), `CURCD_(t−1) = CURCD_(t−2)` for CFO(t−1) and Earn(t−1), and `CURCD_(t+1) = CURCD_t` for CFO(t+1) and Earn(t+1). Thus each numerator and the assets used to scale it share one reported currency. The checks cover all supporting fiscal years 1986–2000, including the raw 1985/2000 asset records needed at the period boundaries. No conversion is attempted where CURCD differs or is missing.

The outcome-year sample remains 1987–1999, while source history is read from 1985–2000. At the 1987 boundary, scaling CFO(t−1) and Earn(t−1) by their own-year average assets reaches total assets in 1985. At the 1999 boundary, CFO(t+1) and Earn(t+1) use 2000 data. The paper describes the sampled outcomes as 1987–1999 and does not say that lead/lag data must lie within those years, so the baseline keeps t=1987 and t=1999 when adjacent source years are available. As a boundary diagnostic, restricting the outcome to t=1988–1998 (so all lead/lag observations also lie in 1987–1999) leaves 29,300 trimmed rows with currency-valid CFO and Earn available, before the eight-observation screen; the baseline does not impose that narrower window.

The sample screens follow the paper’s Table 1 order: (1) fiscal years 1987–1999 with complete current-year assets/scaling inputs, item-123 earnings, CFO, and all five signed working-capital change fields; (2) remove observations outside pooled 1st/99th percentile cutoffs for scaled CFO, item-123 earnings, and scaled change in working capital; (3) require scaled CFO and constructed Earn (CFO + ΔWC) at both *t−1* and *t+1*; and (4) retain firms with at least eight eligible regression observations. “Earnings” in the initial completeness and trimming screens is interpreted as item 123, which the variable definitions identify as Prof; the paper does not explicitly identify that field in the sample-selection paragraph or say whether the trim variable is Prof or constructed Earn. The lead/lag prose is interpreted as requiring constructed Earn because that is the paper's named earnings measure and Table 2 reports Earn(t+1); Table 1's explicit CFO-only row leaves the issue unresolved. These are best-effort interpretations. Percentiles are pooled over eligible 1987–1999 observations because the paper does not state year-specific cutoffs; SAS percentile interpolation defines the cutoff. Missing source fields are not filled with zero. Lead/lag values come from raw adjacent fiscal years; those neighboring years need not independently survive the current-year trims.

## Estimation, aggregation, and unresolved points

The fit is separately estimated by firm using OLS on each firm’s eligible 1987–1999 observations, with a minimum of eight rows. `asreg` is used directly for its established by-group regression and in-memory coefficient results. Firm coefficients and adjusted R-squared are then summarized across firms. Cross-firm t-statistics use the sample standard deviation of the firm estimates divided by the square root of the number of firms, with *N−1* degrees of freedom. No pooled model replaces this coefficient-distribution target. Stata exports the means and p-values using the template format; built-in `ttest` output in the ordinary log retains the paper-reported t statistics with N−1 degrees of freedom.

Unresolved differences include the current local Compustat vintage versus the historical research file, the modern population-code mapping, the precise earnings variable used for truncation, SAS’s percentile interpolation versus the authors’ unstated rule, the unstated estimator and lag/lead denominator dates, and the absence of documented CURRTR arithmetic direction for converting unlike-currency amounts. We will report the realized counts, distributions, and any missing/singular fits without tuning sample choices to match published Table 1 or Table 3 values.

## Execution comparison

The final read-only source scan contained 173,871 `COMP.FUNDA` rows for 1985–2000 and no duplicate `gvkey`–`fyear` keys. The local currency field was nonmissing on every row: 14,365 CAD and 159,506 USD. The 1987–1999 outcome window had 144,162 candidate rows. Of these, 85,486 had nonmissing assets, item-123 earnings, item-308 CFO, and items 302/303. Requiring all five signed working-capital inputs, a positive own-period average-assets denominator, and matching known CURCD for the current denominator yielded 51,955 complete model-input rows; pooled trimming left 49,853.

The currency audit of the raw auxiliary population checked every source-year average-assets pair from 1986 through 2000. The following table gives counts with exact same CURCD, different CURCD, a valid positive same-currency average-assets denominator, scaled CFO, and constructed Earn available. CFO and Earn counts also require their corresponding raw cash-flow inputs.

| Fiscal year scaled | Same CURCD | Different CURCD | Valid average assets | CFO available | Earn available |
| ---: | ---: | ---: | ---: | ---: | ---: |
| 1986 | 8,084 | 5 | 7,162 | 0 | 0 |
| 1987 | 8,491 | 7 | 7,513 | 1,176 | 583 |
| 1988 | 8,607 | 8 | 7,472 | 6,175 | 3,503 |
| 1989 | 8,723 | 5 | 7,374 | 6,500 | 3,718 |
| 1990 | 8,847 | 6 | 7,330 | 6,501 | 3,690 |
| 1991 | 9,126 | 3 | 7,420 | 6,636 | 3,754 |
| 1992 | 9,580 | 11 | 7,624 | 6,879 | 3,840 |
| 1993 | 10,322 | 8 | 8,044 | 7,298 | 4,091 |
| 1994 | 10,881 | 11 | 9,087 | 7,718 | 4,264 |
| 1995 | 11,206 | 15 | 9,476 | 8,209 | 4,422 |
| 1996 | 11,835 | 14 | 10,456 | 9,203 | 4,971 |
| 1997 | 11,825 | 18 | 10,489 | 9,237 | 5,002 |
| 1998 | 11,404 | 13 | 10,060 | 8,942 | 4,805 |
| 1999 | 11,487 | 24 | 10,467 | 9,635 | 5,314 |
| 2000 | 11,339 | 9 | 10,461 | 9,534 | 5,183 |

Matched raw firm-year pairs with missing CURCD were zero; gaps in the annual key are counted separately in the SAS log. Over the full outcome-year candidate population, current t/t−1, past t−1/t−2, and future t+1/t currency mismatches numbered 143, 107, and 145, respectively. These are period-level construction diagnostics, not extra sample screens applied to unrelated amounts. Within the trimmed current sample, the explicit CFO+Earn lead/lag screen retained 32,357 rows across 7,127 firms before the minimum-eight-year requirement; the corresponding CFO-only and CFO+Prof diagnostics retained 37,957 and 37,937 rows. After the eight-year screen, 14,346 firm-years for 1,465 firms remain, with fit lengths of 8 years for 366 firms, 9 for 260, 10 for 248, 11 for 494, and 12 for 97. There are no eligible 1987 outcome rows and 1,004 eligible 1999 rows. Restricting the outcome window to 1988–1998 leaves 12,124 firm-years for 1,291 firms after the eight-year screen.

CURRTR was summarized but not used as a conversion factor. In this source vintage, its observed range was 1.0000–2.5566 for CAD-coded rows and 0.000101698–1.6995 for USD-coded rows. The latter varies even though the guide says the rate is 1.000 when a foreign company reports in U.S. dollars. Alongside the missing arithmetic direction in the provider description, this makes the local field unsuitable as a guessed conversion instruction. The final model sample preserves same-currency observations: 89 CAD firm-years for 10 firms and 14,257 USD firm-years for 1,456 firms; all three checked neighboring CURCD pairs match within every retained observation.

Table 1 reports 59,360, 55,850, 30,317, and 15,234 observations for 1,725 firms (physical PDF pp. 7–8); its footnote reports 481, 1,054, and 190 firms with 8, 9, and 10 observations. The final local fit sample remains below those reported totals. Item 304, 305, and 307 are missing in 65,633, 63,592, and 40,923 of the 144,162 outcome-window rows, respectively; these counts overlap. The local vintage/population and source-field coverage do not isolate a single explanation for the remaining discrepancy, and no sample choice was changed to target the published counts.

For the 1,465 fits, all four coefficient means have two-sided p-values below 0.001, based on 1,464 degrees of freedom. `asreg` estimates one regression per firm; the cell keeps one result per fitted firm, uses native `mean` and `ttest`, and summarizes coefficient and adjusted-R-squared distributions with `estpost tabstat`. Table 3 Panel A reports coefficient means/t-statistics at two decimals. The A3 cross-firm results are:

| Statistic | A3 mean | A3 t | Table 3 mean | Table 3 t |
| --- | ---: | ---: | ---: | ---: |
| Intercept | 0.031 | 14.56 | 0.04 | 23.03 |
| CFO(t−1) | 0.143 | 17.50 | 0.17 | 19.38 |
| CFO(t) | −0.548 | −47.66 | −0.62 | −57.06 |
| CFO(t+1) | 0.122 | 14.29 | 0.09 | 10.38 |
| Adjusted R-squared mean | 0.447 | — | 0.47 | — |

Local coefficient means, quartiles, and adjusted-R-squared distribution are in regression coefficient units. The A3 coefficient lower quartile/median/upper quartile are −0.004/0.027/0.069 for the intercept, −0.023/0.118/0.308 for CFO(t−1), −0.834/−0.569/−0.270 for CFO(t), and −0.057/0.118/0.293 for CFO(t+1). Adjusted-R-squared quartiles are 0.185/0.523/0.751, versus 0.23/0.55/0.80 in the paper. The two-sided p-values in the template export are shown as 0.000; the native tests show all four p-values below 0.001. The paper supplies t-statistics but does not state a one-sided alternative. Table outputs in `.md`, `.rtf`, `.csv`, and `.tex` formats are under `tables/`; the ordinary native logs are under `runtime/`.


In [ ]:
options errorabend;
options validvarname=v7;

libname comp "/storage/sas/base/comp/d_na" access=readonly;
libname out "data";

/* Read only the fields and fiscal years needed for 1987-1999 outcomes and
   the 1985-2000 denominator/CFO history. CURCD is retained to ensure that
   each average-assets denominator is formed within one reporting currency.
   CURRTR is summarized for diagnosis only; no arithmetic direction is
   inferred from its provider definition. */
data work.raw;
    set comp.funda
        (keep=gvkey datadate fyear indfmt datafmt consol popsrc fyr at oancf
              recch invch apalch txach aoloch ibc sich curcd currtr
         where=(fyear >= 1985 and fyear <= 2000 and indfmt="INDL" and datafmt="STD"
                and consol="C" and popsrc="D"));
run;

proc sql noprint;
    select count(*) into :raw_rows trimmed from work.raw;
    select count(*) into :duplicate_pairs trimmed
    from (select gvkey, fyear
          from work.raw
          group by gvkey, fyear
          having count(*) > 1) as d;
quit;
%put NOTE: Filtered annual source rows=&raw_rows.;
%put NOTE: Duplicate gvkey-fyear keys=&duplicate_pairs.;

proc freq data=work.raw;
    tables curcd / missing;
    title "W3122547505 reporting-currency codes in raw auxiliary population";
run;

proc means data=work.raw n nmiss min mean max;
    class curcd;
    var currtr;
    title "W3122547505 provider translation-rate diagnostics (not used for conversion)";
run;

/* A firm-year must be unique before date-keyed lead/lag construction. Do not
   hide duplicates with NODUPKEY or select a vintage without evidence. */
%if &duplicate_pairs > 0 %then %do;
    %put ERROR: Resolve the reported duplicate gvkey-fyear keys before estimation.;
    %abort cancel;
%end;

proc sort data=work.raw;
    by gvkey fyear;
run;

/* Shift by fiscal year so gaps remain gaps. Retain adjacent earnings inputs
   as well as CFO because the prose sample rule mentions both. */
data work.lag1(keep=gvkey fyear at_lag1 curcd_lag1
                    cfo_lag_raw ibc_lag_raw d_wc_lag_raw);
    set work.raw(keep=gvkey fyear at oancf ibc recch invch apalch txach aoloch curcd);
    fyear = fyear + 1;
    at_lag1 = at;
    curcd_lag1 = curcd;
    cfo_lag_raw = oancf;
    ibc_lag_raw = ibc;
    if nmiss(recch, invch, apalch, txach, aoloch)=0 then
        d_wc_lag_raw = -(recch + invch + apalch + txach + aoloch);
    else d_wc_lag_raw = .;
run;

data work.lag2(keep=gvkey fyear at_lag2 curcd_lag2);
    set work.raw(keep=gvkey fyear at curcd);
    fyear = fyear + 2;
    at_lag2 = at;
    curcd_lag2 = curcd;
run;

data work.lead1(keep=gvkey fyear at_lead1 curcd_lead1
                     cfo_lead_raw ibc_lead_raw d_wc_lead_raw);
    set work.raw(keep=gvkey fyear at oancf ibc recch invch apalch txach aoloch curcd);
    fyear = fyear - 1;
    at_lead1 = at;
    curcd_lead1 = curcd;
    cfo_lead_raw = oancf;
    ibc_lead_raw = ibc;
    if nmiss(recch, invch, apalch, txach, aoloch)=0 then
        d_wc_lead_raw = -(recch + invch + apalch + txach + aoloch);
    else d_wc_lead_raw = .;
run;

proc sort data=work.lag1; by gvkey fyear; run;
proc sort data=work.lag2; by gvkey fyear; run;
proc sort data=work.lead1; by gvkey fyear; run;

/* Check the same-period average-assets denominator for every raw year used
   by current, past, and future variables: 1986-2000. This includes 1986
   history for 1987 outcomes and 2000 history for 1999 outcomes. */
data work.prior_period(keep=gvkey fyear at_prior curcd_prior);
    set work.raw(keep=gvkey fyear at curcd);
    fyear = fyear + 1;
    at_prior = at;
    curcd_prior = curcd;
run;
proc sort data=work.prior_period; by gvkey fyear; run;

data work.currency_periods;
    merge work.raw(in=current_row)
          work.prior_period(in=prior_row);
    by gvkey fyear;
    if current_row and fyear >= 1986 and fyear <= 2000;
    has_prior_row = prior_row;
    curcd_same_as_prior = 0;
    if has_prior_row and not missing(curcd) and not missing(curcd_prior)
       and curcd = curcd_prior then curcd_same_as_prior = 1;
    avg_at_period = .;
    if curcd_same_as_prior and has_prior_row and nmiss(at, at_prior)=0 then do;
        avg_at_period = (at + at_prior) / 2;
        if avg_at_period <= 0 then avg_at_period = .;
    end;
    if not missing(avg_at_period) then do;
        cfo_scaled_period = oancf / avg_at_period;
        if nmiss(recch, invch, apalch, txach, aoloch)=0 then do;
            d_wc_period = -(recch + invch + apalch + txach + aoloch);
            earn_scaled_period = (oancf + d_wc_period) / avg_at_period;
        end;
        else do;
            d_wc_period = .;
            earn_scaled_period = .;
        end;
    end;
    else do;
        cfo_scaled_period = .;
        d_wc_period = .;
        earn_scaled_period = .;
    end;
run;

proc sql;
    title "W3122547505 CURCD and own-period denominator checks over raw auxiliary years";
    select fyear as scaled_fyear,
           count(*) as raw_firm_years,
           sum(has_prior_row=0) as no_prior_year_record,
           sum(has_prior_row=1 and (missing(curcd) or missing(curcd_prior))) as missing_currency_pair,
           sum(has_prior_row=1 and not missing(curcd) and not missing(curcd_prior)
               and curcd=curcd_prior) as same_currency_pair,
           sum(has_prior_row=1 and not missing(curcd) and not missing(curcd_prior)
               and curcd ne curcd_prior) as different_currency_pair,
           sum(has_prior_row=1 and nmiss(at,at_prior)=0) as assets_pair_fields_present,
           sum(not missing(avg_at_period)) as currency_valid_average_assets,
           sum(not missing(cfo_scaled_period)) as currency_valid_cfo,
           sum(not missing(earn_scaled_period)) as currency_valid_earn
    from work.currency_periods
    group by fyear
    order by fyear;
quit;

data work.anchored;
    merge work.raw(in=anchor)
          work.lag1
          work.lag2
          work.lead1;
    by gvkey fyear;
    if anchor and fyear >= 1987 and fyear <= 1999;

    /* Paper item 302+303+304+305+307 are signed cash-flow changes. */
    if nmiss(recch, invch, apalch, txach, aoloch)=0 then
        d_wc = -(recch + invch + apalch + txach + aoloch);
    else d_wc = .;

    /* Make a denominator available only when both source assets share CURCD.
       No average is formed across unlike-currency amounts. */
    curcd_ok_current = (not missing(curcd) and not missing(curcd_lag1)
                        and curcd = curcd_lag1);
    curcd_ok_lag = (not missing(curcd_lag1) and not missing(curcd_lag2)
                    and curcd_lag1 = curcd_lag2);
    curcd_ok_lead = (not missing(curcd) and not missing(curcd_lead1)
                     and curcd = curcd_lead1);

    avg_at = .;
    avg_at_lag = .;
    avg_at_lead = .;
    if curcd_ok_current and nmiss(at, at_lag1)=0 then do;
        avg_at = (at + at_lag1) / 2;
        if avg_at <= 0 then avg_at = .;
    end;
    if curcd_ok_lag and nmiss(at_lag1, at_lag2)=0 then do;
        avg_at_lag = (at_lag1 + at_lag2) / 2;
        if avg_at_lag <= 0 then avg_at_lag = .;
    end;
    if curcd_ok_lead and nmiss(at, at_lead1)=0 then do;
        avg_at_lead = (at + at_lead1) / 2;
        if avg_at_lead <= 0 then avg_at_lead = .;
    end;

    if avg_at > 0 then do;
        cfo_scaled = oancf / avg_at;
        prof_scaled = ibc / avg_at;
        d_wc_scaled = d_wc / avg_at;
        earn_scaled = cfo_scaled + d_wc_scaled;
    end;
    else do;
        cfo_scaled = .;
        prof_scaled = .;
        d_wc_scaled = .;
    end;

    if avg_at_lag > 0 and not missing(cfo_lag_raw) then
        cfo_lag = cfo_lag_raw / avg_at_lag;
    else cfo_lag = .;
    if avg_at_lag > 0 and not missing(ibc_lag_raw) then
        prof_lag = ibc_lag_raw / avg_at_lag;
    else prof_lag = .;
    if avg_at_lag > 0 and not missing(cfo_lag_raw) and not missing(d_wc_lag_raw) then
        earn_lag = (cfo_lag_raw + d_wc_lag_raw) / avg_at_lag;
    else earn_lag = .;

    if avg_at_lead > 0 and not missing(cfo_lead_raw) then
        cfo_lead = cfo_lead_raw / avg_at_lead;
    else cfo_lead = .;
    if avg_at_lead > 0 and not missing(ibc_lead_raw) then
        prof_lead = ibc_lead_raw / avg_at_lead;
    else prof_lead = .;
    if avg_at_lead > 0 and not missing(cfo_lead_raw) and not missing(d_wc_lead_raw) then
        earn_lead = (cfo_lead_raw + d_wc_lead_raw) / avg_at_lead;
    else earn_lead = .;

    /* Availability indicators count complete source fields, not fabricated
       scaled values, for the pre-currency audit comparison. */
    cfo_lag_source_available = (not missing(cfo_lag_raw)
                                and nmiss(at_lag1, at_lag2)=0);
    earn_lag_source_available = (not missing(cfo_lag_raw)
                                 and not missing(d_wc_lag_raw)
                                 and nmiss(at_lag1, at_lag2)=0);
    cfo_lead_source_available = (not missing(cfo_lead_raw)
                                 and nmiss(at, at_lead1)=0);
    earn_lead_source_available = (not missing(cfo_lead_raw)
                                  and not missing(d_wc_lead_raw)
                                  and nmiss(at, at_lead1)=0);

    complete_reported_fields = (nmiss(at, ibc, oancf, recch, invch)=0);
    source_inputs_precur = (nmiss(at, at_lag1, ibc, oancf,
                                  recch, invch, apalch, txach,
                                  aoloch)=0);
    complete_model_inputs = (nmiss(at, ibc, oancf, recch, invch,
                                   apalch, txach, aoloch,
                                   avg_at, d_wc_scaled,
                                   cfo_scaled, prof_scaled)=0);
run;

proc sql;
    title "W3122547505 source coverage and pre-trim sample counts";
    select count(*) as outcome_year_rows,
           sum(complete_reported_fields) as reported_fields_complete,
           sum(source_inputs_precur) as source_inputs_precur,
           sum(complete_model_inputs) as all_model_inputs_complete,
           count(distinct gvkey) as firms_in_outcome_window
    from work.anchored;
quit;

proc sql;
    title "W3122547505 currency status for each model denominator window";
    select count(*) as outcome_year_rows,
           sum(curcd_ok_current) as current_t_and_tminus1_same,
           sum(not missing(curcd) and not missing(curcd_lag1)
               and curcd ne curcd_lag1) as current_t_and_tminus1_different,
           sum(curcd_ok_lag) as past_tminus1_and_tminus2_same,
           sum(not missing(curcd_lag1) and not missing(curcd_lag2)
               and curcd_lag1 ne curcd_lag2) as past_tm1_tm2_different,
           sum(curcd_ok_lead) as future_tplus1_and_t_same,
           sum(not missing(curcd) and not missing(curcd_lead1)
               and curcd ne curcd_lead1) as future_tplus1_and_t_different
    from work.anchored;
quit;

proc sql;
    title "W3122547505 field missingness in outcome-year population";
    select count(*) as outcome_year_rows,
           sum(missing(at)) as missing_assets,
           sum(missing(ibc)) as missing_item123_earnings,
           sum(missing(oancf)) as missing_item308_cfo,
           sum(missing(recch)) as missing_item302,
           sum(missing(invch)) as missing_item303,
           sum(missing(apalch)) as missing_item304,
           sum(missing(txach)) as missing_item305,
           sum(missing(aoloch)) as missing_item307,
           sum(missing(avg_at)) as missing_current_avg_assets,
           sum(missing(d_wc_scaled)) as missing_scaled_delta_wc
    from work.anchored;
quit;

data work.complete;
    set work.anchored;
    if complete_model_inputs;
run;

/* The paper does not state whether percentiles are year-specific. Use pooled
   SAS 1st/99th percentile cutoffs for the scaled current-year variables. */
proc univariate data=work.complete noprint;
    var cfo_scaled prof_scaled d_wc_scaled;
    output out=work.cutoffs
        pctlpts=1 99
        pctlpre=cfo_ prof_ dwc_;
run;

data work.trimmed;
    if _n_=1 then set work.cutoffs;
    set work.complete;
    if cfo_scaled >= cfo_1 and cfo_scaled <= cfo_99
       and prof_scaled >= prof_1 and prof_scaled <= prof_99
       and d_wc_scaled >= dwc_1 and d_wc_scaled <= dwc_99;
run;

proc sql;
    title "W3122547505 sample attrition by Table 1 screen";
    select "Complete variables before trimming" as stage length=40,
           count(*) as firm_years,
           count(distinct gvkey) as firms
    from work.complete
    union all
    select "After pooled 1/99 percentile trimming" as stage length=40,
           count(*) as firm_years,
           count(distinct gvkey) as firms
    from work.trimmed;
quit;

proc sql;
    title "W3122547505 lead/lag availability after current-row trimming";
    select count(*) as trimmed_firm_years,
           sum(cfo_lag_source_available) as lag_cfo_inputs_present,
           sum(not missing(cfo_lag)) as lag_cfo_currency_valid,
           sum(earn_lag_source_available) as lag_earn_inputs_present,
           sum(not missing(earn_lag)) as lag_earn_currency_valid,
           sum(cfo_lead_source_available) as lead_cfo_inputs_present,
           sum(not missing(cfo_lead)) as lead_cfo_currency_valid,
           sum(earn_lead_source_available) as lead_earn_inputs_present,
           sum(not missing(earn_lead)) as lead_earn_currency_valid,
           sum(cfo_lag_source_available and not missing(cfo_scaled)
               and cfo_lead_source_available and earn_lag_source_available
               and not missing(earn_scaled) and earn_lead_source_available)
               as all_cfo_earn_inputs_precur,
           sum(nmiss(cfo_lag,cfo_scaled,cfo_lead,
                     earn_lag,earn_scaled,earn_lead)=0) as all_cfo_earn_currency_valid,
           sum(missing(cfo_lag)) as missing_scaled_lag_cfo,
           sum(missing(cfo_lead)) as missing_scaled_lead_cfo,
           sum(missing(earn_lag)) as missing_scaled_lag_earn,
           sum(missing(earn_lead)) as missing_scaled_lead_earn,
           sum(missing(prof_lag)) as missing_scaled_lag_prof,
           sum(missing(prof_lead)) as missing_scaled_lead_prof
    from work.trimmed;
quit;

/* Compare Table 1's explicit CFO lead/lag row with the surrounding prose
   requiring past/future cash flows and earnings. Earn is CFO + delta WC. */
proc sql;
    title "W3122547505 lead/lag eligibility alternatives and period boundaries";
    select "CFO only, outcome 1987-1999" as screen length=48,
           count(*) as firm_years, count(distinct gvkey) as firms
    from work.trimmed
    where nmiss(cfo_lag, cfo_scaled, cfo_lead)=0
    union all
    select "CFO plus Earn, outcome 1987-1999" as screen length=48,
           count(*) as firm_years, count(distinct gvkey) as firms
    from work.trimmed
    where nmiss(cfo_lag, cfo_scaled, cfo_lead, earn_lag, earn_scaled, earn_lead)=0
    union all
    select "CFO plus Prof, outcome 1987-1999" as screen length=48,
           count(*) as firm_years, count(distinct gvkey) as firms
    from work.trimmed
    where nmiss(cfo_lag, cfo_scaled, cfo_lead, prof_lag, prof_scaled, prof_lead)=0
    union all
    select "CFO plus Earn, all years inside 1987-1999" as screen length=48,
           count(*) as firm_years, count(distinct gvkey) as firms
    from work.trimmed
    where fyear >= 1988 and fyear <= 1998
      and nmiss(cfo_lag, cfo_scaled, cfo_lead, earn_lag, earn_scaled, earn_lead)=0;
quit;

/* Preserve the observed outcome-year window and adjacent raw-year inputs.
   Follow the p. 8 sample prose by requiring lead/lag Earn in addition to CFO.
   Neighbor years need not independently pass the current-year trimming rule. */
data work.with_leads;
    set work.trimmed;
    if nmiss(cfo_lag, cfo_scaled, cfo_lead,
             earn_lag, earn_scaled, earn_lead)=0;
run;

proc sql;
    title "W3122547505 post lead/lag CFO and Earn screen";
    select count(*) as firm_years,
           count(distinct gvkey) as firms
    from work.with_leads;
quit;

proc sql;
    create table work.firm_counts as
    select gvkey, count(*) as n_firm_years
    from work.with_leads
    group by gvkey;

    create table work.fit_sample as
    select a.gvkey, a.fyear, a.curcd,
           a.curcd_lag1, a.curcd_lag2, a.curcd_lead1, a.d_wc_scaled,
           a.cfo_lag, a.cfo_scaled as cfo_cur, a.cfo_lead,
           b.n_firm_years
    from work.with_leads as a
    inner join work.firm_counts as b
      on a.gvkey=b.gvkey
    where b.n_firm_years >= 8
    order by a.gvkey, a.fyear;
quit;

proc sql;
    title "W3122547505 firm-specific regression sample";
    select count(*) as firm_years,
           count(distinct gvkey) as firms,
           min(n_firm_years) as min_years_per_firm,
           max(n_firm_years) as max_years_per_firm
    from work.fit_sample;
quit;

proc sql;
    title "W3122547505 firm-specific sample observation-count distribution";
    select n_firm_years, count(distinct gvkey) as firms
    from work.fit_sample
    group by n_firm_years
    order by n_firm_years;
quit;

proc sql;
    title "W3122547505 outcome-year boundary observations";
    select fyear, count(*) as firm_years, count(distinct gvkey) as firms
    from work.fit_sample
    group by fyear
    order by fyear;
quit;

data out.firm_specific;
    set work.fit_sample;
    label d_wc_scaled="Change in working capital / average assets"
          cfo_lag="CFO(t-1) / average assets(t-1)"
          cfo_cur="CFO(t) / average assets(t)"
          cfo_lead="CFO(t+1) / average assets(t+1)"
          curcd="CURCD at outcome year"
          curcd_lag1="CURCD at outcome year minus one"
          curcd_lag2="CURCD at outcome year minus two"
          curcd_lead1="CURCD at outcome year plus one"
          n_firm_years="Eligible annual observations for firm";
run;

proc sql;
    title "W3122547505 retained sample currency check (same-currency records stay eligible)";
    select curcd,
           count(*) as firm_years,
           count(distinct gvkey) as firms,
           sum(curcd=curcd_lag1) as current_and_prior_same,
           sum(curcd_lag1=curcd_lag2) as prior_and_two_prior_same,
           sum(curcd=curcd_lead1) as current_and_following_same
    from out.firm_specific
    group by curcd
    order by curcd;
quit;

proc sql;
    title "W3122547505 final sample key check";
    select count(*) as firm_years,
           count(distinct gvkey) as firms,
           count(distinct cats(gvkey, "-", put(fyear, 4.))) as firm_year_keys
    from out.firm_specific;
quit;

title;
